<font color='#007CE5'>**CÓDIGO DE LIMPIEZA DE BASE DE DATOS OFICIAL DE LOS MOSSOS D'ESQUADRA**</font>

In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
df = pd.read_csv("FetsPenals.csv")

df

,Mes,Nom mes,Any,Regió Policial (RP),Àrea Bàsica Policial (ABP),Títol Codi Penal,Tipus de fet,Coneguts,Resolts,Detencions
0,1,gener,2011,RP Central,ABP Osona,De les falsedats,Falsedats documentals,3,2,1
1,1,gener,2011,RP Central,ABP Osona,De les lesions,Lesions,4,5,1
2,1,gener,2011,RP Central,ABP Osona,De les tortures i altres delictes contra la in...,Tracte degradant / vexatori,11,13,6
3,1,gener,2011,RP Central,ABP Osona,Delictes contra el patrimoni i contra l’ordre ...,Apropiació indeguda,2,2,0
4,1,gener,2011,RP Central,ABP Osona,Delictes contra el patrimoni i contra l’ordre ...,Danys,7,1,0
...,...,...,...,...,...,...,...,...,...,...
326022,6,juny,2025,RP Camp de Tarragona,ABP Tarragonès,Delictes contra la seguretat viària,Conduir sense permís,3,3,1
326023,6,juny,2025,RP Camp de Tarragona,ABP Tarragonès,Delictes contra la seguretat viària,Conduir sota els efectes d'alcohol i drogues,5,5,1
326024,6,juny,2025,RP Camp de Tarragona,ABP Tarragonès,Delictes contra la seguretat viària,Negativa a sotmetre's a les proves,2,2,0
326025,6,juny,2025,RP Camp de Tarragona,ABP Tarragonès,Delictes contra les relacions familiars,Contra els drets i deures familiars,1,1,0


In [2]:
null_df = df.isnull().sum()
null_df

Mes                           0
Nom mes                       0
Any                           0
Regió Policial (RP)           0
Àrea Bàsica Policial (ABP)    0
Títol Codi Penal              7
Tipus de fet                  7
Coneguts                      0
Resolts                       0
Detencions                    0
dtype: int64

In [3]:
# Filtrar filas donde hay valores nulos en una de las dos columnas
df_nulos = df[df["Títol Codi Penal"].isna() | df["Tipus de fet"].isna()]

print(df_nulos)

        Mes   Nom mes   Any         Regió Policial (RP)  \
262490    5      maig  2023  RP Metropolitana Barcelona   
264397    6      juny  2023  RP Metropolitana Barcelona   
265258    6      juny  2023      RP  Metropolitana Nord   
267585    8     agost  2023                  RP Central   
267755    8     agost  2023                   RP Girona   
274434   11  novembre  2023      RP  Metropolitana Nord   
276528   12  desembre  2023                   RP Ponent   

                   Àrea Bàsica Policial (ABP) Títol Codi Penal Tipus de fet  \
262490                     ABP Sants-Montjuïc              NaN          NaN   
264397                     ABP Sants-Montjuïc              NaN          NaN   
265258                           ABP Sabadell              NaN          NaN   
267585                              ABP Bages              NaN          NaN   
267755          ABP Gironès - Pla de l'Estany              NaN          NaN   
274434                           ABP Badalona        

In [ ]:
def rellenar_con_moda(df, col_text, col_group):

    fill_map = {}
    for g in df[col_group].unique():
        moda = df.loc[df[col_group] == g, col_text].mode()
        fill_map[g] = moda[0] if not moda.empty else "Desconegut"

    df[col_text] = df.apply(
        lambda row: fill_map[row[col_group]] if pd.isnull(row[col_text]) else row[col_text],
        axis=1
    )
    return df

df = rellenar_con_moda(df, col_text="Títol Codi Penal", col_group="Àrea Bàsica Policial (ABP)")

df = rellenar_con_moda(df, col_text="Tipus de fet", col_group="Àrea Bàsica Policial (ABP)")


In [5]:
df.dtypes

Mes                            int64
Nom mes                       object
Any                            int64
Regió Policial (RP)           object
Àrea Bàsica Policial (ABP)    object
Títol Codi Penal              object
Tipus de fet                  object
Coneguts                       int64
Resolts                        int64
Detencions                     int64
dtype: object

In [6]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('à', 'a').str.replace('ó', 'o').str.replace('í', 'i').str.replace('(', '').str.replace(')', '').str.replace("regio_policial_rp", "regio_policial").str.replace("area_basica_policial_abp", "area_basica_policial")


In [7]:
df.isnull().sum()

mes                     0
nom_mes                 0
any                     0
regio_policial          0
area_basica_policial    0
titol_codi_penal        0
tipus_de_fet            0
coneguts                0
resolts                 0
detencions              0
dtype: int64

In [8]:
df['id'] = range(1, len(df) + 1)

df["coneguts"].sum()

7657555

In [9]:
datostf = sorted([str(x) for x in df["tipus_de_fet"].unique()])

for i in range(len(datostf)):
    for j in range(i+1, len(datostf)):
        score = fuzz.ratio(datostf[i], datostf[j])
        if score > 85:
            print(f"{score:.1f}% - '{datostf[i]}' vs '{datostf[j]}'")


97.8% - 'Administració desleial' vs 'Administració deslleial'
87.5% - 'Calumnia' vs 'Calúmnia'
98.1% - 'Contra les institucions de l'Estat i divisió de poders' vs 'Contra les institucions de l'estat i divisió de poders'
97.7% - 'Defraudacions de fluid elèctric i anàlogues' vs 'Defraudacions de fluïd elèctric i anàlogues'
96.3% - 'Fraus i exaccions il·legals' vs 'Fraus i execcions il·legals'
89.1% - 'Infidelitat custòdia documents i violació secrets' vs 'Infidelitat en la custòdia de documents i violació de secrets'
96.7% - 'Reunió, manifestació il·licita' vs 'Reunió, manifestació il·lícita'
95.5% - 'Tràfic d'èssers humans' vs 'Tràfic d'éssers humans'
97.7% - 'Usurpació de funcions publiques i intrusisme' vs 'Usurpació de funcions públiques i intrusisme'


In [ ]:
df["tipus_de_fet"] = df["tipus_de_fet"].replace("Calumnia", "Calúmnia").replace("Contra les institucions de l'estat i divisió de poders", "Contra les institucions de l'Estat i divisió de poders").replace('Defraudacions de fluïd elèctric i anàlogues', 'Defraudacions de fluid elèctric i anàlogues').replace('Fraus i execcions il·legals', 'Fraus i exaccions il·legals').replace('Reunió, manifestació il·licita', 'Reunió, manifestació il·lícita').replace("Tràfic d'èssers humans", "Tràfic d'éssers humans").replace('Usurpació de funcions publiques i intrusisme', 'Usurpació de funcions públiques i intrusisme').replace('Administració desleial', 'Administració deslleial').replace('Infidelitat custòdia documents i violació secrets', 'Infidelitat en la custòdia de documents i violació de secrets')


In [11]:
datostcp = sorted([str(x) for x in df["titol_codi_penal"].unique()])

for i in range(len(datostcp)):
    for j in range(i+1, len(datostcp)):
        score = fuzz.ratio(datostcp[i], datostcp[j])
        if score > 85:
            print(f"{score:.1f}% - '{datostcp[i]}' vs '{datostcp[j]}'")

89.7% - 'Delictes contra l'ordre públic' vs 'Faltes contra l'ordre públic'


In [12]:
datosrp = sorted([str(x) for x in df["regio_policial"].unique()])

for i in range(len(datosrp)):
    for j in range(i+1, len(datosrp)):
        score = fuzz.ratio(datosrp[i], datosrp[j])
        if score > 85:
            print(f"{score:.1f}% - '{datosrp[i]}' vs '{datosrp[j]}'")

97.7% - 'RP  Metropolitana Nord' vs 'RP Metropolitana Nord'
85.7% - 'RP  Metropolitana Nord' vs 'RP Metropolitana Sud'
87.8% - 'RP Metropolitana Nord' vs 'RP Metropolitana Sud'


In [13]:
df["regio_policial"] = df["regio_policial"].replace({'RP ':''}, regex=True)
df["regio_policial"] = df["regio_policial"].replace(" Metropolitana Nord", "Metropolitana Nord")

In [14]:
datosmes = sorted([str(x) for x in df["nom_mes"].unique()])

for i in range(len(datosmes)):
    for j in range(i+1, len(datosmes)):
        score = fuzz.ratio(datosmes[i], datosmes[j])
        if score > 85:
            print(f"{score:.1f}% - '{datosmes[i]}' vs '{datosmes[j]}'")

In [15]:
datosabp = sorted([str(x) for x in df["area_basica_policial"].unique()])

for i in range(len(datosabp)):
    for j in range(i+1, len(datosabp)):
        score = fuzz.ratio(datosabp[i], datosabp[j])
        if score > 85:
            print(f"{score:.1f}% - '{datosabp[i]}' vs '{datosabp[j]}'")

92.3% - 'ABP Cerdanya' vs 'ABP Cerdanyola'
88.5% - 'ABP Sant Boi de Llobregat' vs 'ABP Sant Feliu de Llobregat'


In [16]:
df["tipus_de_fet"].unique()

array(['Falsedats documentals', 'Lesions', 'Tracte degradant / vexatori',
       'Apropiació indeguda', 'Danys', 'Furt', 'Robatori amb força',
       'Robatori amb força interior vehicle',
       'Robatori amb violència i/o intimidació',
       "Robatori i furt d'us de vehicle",
       'Acusació, denúncia falsa i simulació de delictes',
       'Realització arbitrària del propi dret', 'Trencament de condemna',
       "Atemptat a autoritat, agents de l' autoritat i resistència i desobediència",
       'Entrada a vivenda aliena', 'Amenaces', 'Detenció il·legal',
       'Agressions sexuals i Abusos sexuals', 'Conduir sense permís',
       "Conduir sota els efectes d'alcohol i drogues",
       "Negativa a sotmetre's a les proves",
       'Contra els drets i deures familiars', 'Contra la salut pública',
       'Faltes contra el patrimoni',
       'Faltes contra els interessos generals',
       "Faltes contra l'ordre públic", 'Faltes contra les persones',
       "De la usurpació de l'estat ci

In [17]:
df["area_basica_policial"] = df["area_basica_policial"].replace({'ABP ':''}, regex=True)

In [18]:
df_filtrado = df[(df["any"] >= 2020) & (df["any"] <= 2025)]

In [ ]:
df_filtrado = df_filtrado.reset_index(drop=True)

# df_filtrado

,mes,nom_mes,any,regio_policial,area_basica_policial,titol_codi_penal,tipus_de_fet,coneguts,resolts,detencions,id
0,9,setembre,2024,Central,Osona,De les falsedats,De la usurpació de l'estat civil,3,4,0,130
1,9,setembre,2024,Central,Osona,De les falsedats,Falsedats documentals,4,3,0,209
2,9,setembre,2024,Central,Osona,De les falsedats,Falsificació de moneda i efectes timbrats,1,0,0,641
3,9,setembre,2024,Central,Osona,De les lesions,Lesions,23,23,7,719
4,4,abril,2025,Regió Virtual,Virtual,De les falsedats,De la usurpació de l'estat civil,452,9,0,757
...,...,...,...,...,...,...,...,...,...,...,...
133611,6,juny,2025,Camp de Tarragona,Tarragonès,Delictes contra la seguretat viària,Conduir sense permís,3,3,1,326023
133612,6,juny,2025,Camp de Tarragona,Tarragonès,Delictes contra la seguretat viària,Conduir sota els efectes d'alcohol i drogues,5,5,1,326024
133613,6,juny,2025,Camp de Tarragona,Tarragonès,Delictes contra la seguretat viària,Negativa a sotmetre's a les proves,2,2,0,326025
133614,6,juny,2025,Camp de Tarragona,Tarragonès,Delictes contra les relacions familiars,Contra els drets i deures familiars,1,1,0,326026


In [ ]:
# df_filtrado.to_csv(r'C:\Users\Usuario\Desktop\2. DATOS\Proyecto\Crimenes\crims.csv')

In [21]:
tipusdefet = df["tipus_de_fet"].unique()
tipusdefet_df = pd.DataFrame(tipusdefet, columns=["tipus_de_fet"])
# tipusdefet_df.to_csv('tipus_de_fet.csv', index=False, sep=';', encoding='utf-8')

In [22]:
# TOTAL DE CONOCIDOS 2019

df_filtrado = df[df['any'] == 2019]

df_filtrado
suma_columna = df_filtrado['coneguts'].sum()

suma_columna

571913

In [23]:
# TOTAL DE CONOCIDOS 2018

df_filtrado = df[df['any'] == 2018]

df_filtrado
suma_columna = df_filtrado['coneguts'].sum()

suma_columna


549593